# Apply a component by index

Adapted from Qualtran v0.7.0, Copyright 2024 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.text_drawer import draw_text
from flagquantum.dynamic import DynamicCircuit

## `ApplyLthBloq`
A SELECT operation that executes one of a list of bloqs $U_l$ based on a quantum index:

$$
\mathrm{SELECT} = \sum_{l}|l \rangle \langle l| \otimes U_l
$$

This bloq uses the unary iteration scheme to apply `ops[selection]` controlled on an optional
single-bit `control` register.

#### Parameters
 - `ops`: NDArray of bloqs. Each bloq must have identical registers that are all THRU.
 - `selection_regs`: List of selection registers, defaults to N-D selection index based on `ops`.
 - `control_val`: If provided, a singly controlled gate is constructed. 

#### Registers
 - `selection`: The indices of the bloq in `ops` to execute.
 - `control`: The control bit if specified above.
 - `[user_spec]`: The output registers of the bloqs in `ops`. 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662). Babbush et al. (2018). Section III.A. and Figure 7.


In [ ]:
from examples.algorithms.lth_component_catalog import (
    IndexedComponentExample, indexed_component_example_call_graph,
)

### Example Instances

In [ ]:
op = IndexedComponentExample()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("T/H/Z/X on a shared target", op.signature, {})])))

### Call Graph

In [ ]:
graph = indexed_component_example_call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)
resources = estimate_component_resources(op.resource_model())
print("Declared source resources:", resources)
print("Source T equivalents at 11 T/rotation:", resources.gates.total_t_count())

### Compile the selected component

One control and a big-endian two-bit selector choose T, H, Z, or X on the same target wire. The two parent work wires start in the zero state. The emitted controlled phase and controlled Hadamard are shown as native leaves. The estimates above separately count their source rotation costs.

In [ ]:
circuit = DynamicCircuit(op.qubit_count)
circuit.append(op)
ir = circuit.to_ir()
counts = Counter(gate.name for gate in ir.instructions)
print(draw_text(ir, show_all_qubits=True))
print("Emitted primitive instructions:", dict(counts))
assert counts["t"] + counts["tdg"] == 12
assert counts["measure"] == 3
circuit.append(IndexedComponentExample(adjoint=True))
assert len(circuit.to_ir().instructions) == 2 * len(ir.instructions)